# Case Study 6: Character-Level Text Generation — Shakespeare

## RNN vs LSTM vs GRU Comprehensive Comparison

---

### Objective
Train character-level language models on Shakespeare's complete works and generate new text in his style.

### What You Will Learn
1. Character-level language modeling (next-character prediction)
2. Temperature-controlled sampling for text generation
3. Autoregressive generation with RNN/LSTM/GRU
4. How training progression affects generation quality
5. Perplexity as a language model metric

### Dataset
- **Shakespeare Text**: 1.1M characters, ~40,000 lines from plays and sonnets
- 65 unique characters (letters, punctuation, whitespace)

---
## 1. Environment Setup

In [ ]:
import random
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
COLORS = {'RNN': '#e74c3c', 'LSTM': '#2ecc71', 'GRU': '#3498db'}

---
## 2. Data Loading & EDA

In [ ]:
with open('Text_and_NLP/shakespeare_text/shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Total characters: {len(text):,}')
print(f'Total lines: {len(text.splitlines()):,}')
print(f'Total words: {len(text.split()):,}')
print(f'\nFirst 500 characters:')
print('='*60)
print(text[:500])

In [ ]:
# Character analysis
chars = sorted(set(text))
VOCAB_SIZE = len(chars)
print(f'Unique characters: {VOCAB_SIZE}')
print(f'Characters: {repr("".join(chars))}')

# Character frequency
from collections import Counter
char_counts = Counter(text)
char_freq = pd.DataFrame([
    {'char': repr(c) if c in '\n\t ' else c, 'count': cnt, 'pct': cnt/len(text)*100}
    for c, cnt in char_counts.most_common()
])

fig, ax = plt.subplots(figsize=(16, 5))
top_30 = char_freq.head(30)
ax.bar(range(len(top_30)), top_30['pct'], color='#3498db', edgecolor='white')
ax.set_xticks(range(len(top_30)))
ax.set_xticklabels(top_30['char'], fontsize=9)
ax.set_title('Top 30 Character Frequencies', fontsize=14)
ax.set_ylabel('Frequency (%)')
ax.set_xlabel('Character')
plt.tight_layout()
plt.show()

In [ ]:
# Character type breakdown
lowercase = sum(1 for c in text if c.islower())
uppercase = sum(1 for c in text if c.isupper())
spaces = text.count(' ')
newlines = text.count('\n')
punct = sum(1 for c in text if not c.isalnum() and c not in ' \n\t')
digits = sum(1 for c in text if c.isdigit())

categories = ['Lowercase', 'Uppercase', 'Spaces', 'Newlines', 'Punctuation', 'Digits']
values = [lowercase, uppercase, spaces, newlines, punct, digits]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(categories, values, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c'])
ax.set_title('Character Type Distribution', fontsize=14)
ax.set_xlabel('Count')
for i, v in enumerate(values):
    ax.text(v + 1000, i, f'{v:,} ({v/len(text)*100:.1f}%)', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 3. Preprocessing

**Strategy:**
- Create char-to-index and index-to-char mappings
- Input: sequence of 100 characters → Target: same sequence shifted by 1
- Train/Val split: 90%/10% by position

In [ ]:
# Character mappings
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

# Encode entire text
encoded = np.array([char_to_idx[c] for c in text], dtype=np.int64)
print(f'Encoded shape: {encoded.shape}')
print(f'First 20 encoded: {encoded[:20]}')
print(f'Decoded back: {repr("".join(idx_to_char[i] for i in encoded[:20]))}')

In [ ]:
SEQ_LENGTH = 100  # Input sequence length

class CharDataset(Dataset):
    """Character-level dataset: input is seq of chars, target is shifted by 1."""
    def __init__(self, data, seq_length):
        self.data = data
        self.seq_length = seq_length
        self.n_samples = len(data) - seq_length
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        x = torch.LongTensor(self.data[idx:idx+self.seq_length])
        y = torch.LongTensor(self.data[idx+1:idx+self.seq_length+1])
        return x, y

# Train/Val split (90/10 by position)
split = int(len(encoded) * 0.9)
train_data = encoded[:split]
val_data = encoded[split:]

BATCH_SIZE = 64
train_dataset = CharDataset(train_data, SEQ_LENGTH)
val_dataset = CharDataset(val_data, SEQ_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, drop_last=True)

print(f'Training sequences:   {len(train_dataset):,}')
print(f'Validation sequences: {len(val_dataset):,}')
print(f'Batches per epoch:    {len(train_loader):,}')

In [ ]:
# Visualize a sample
x_sample, y_sample = train_dataset[0]
input_text = ''.join(idx_to_char[i.item()] for i in x_sample)
target_text = ''.join(idx_to_char[i.item()] for i in y_sample)
print('Input (first 80 chars):', repr(input_text[:80]))
print('Target (first 80 chars):', repr(target_text[:80]))
print('\nNote: target is input shifted by 1 character')

---
## 4. Model Architecture

### Architecture
```
Input char indices: (batch, seq_len=100)
        |
  [Embedding]  → (batch, seq_len, embed_dim)
        |
  [RNN/LSTM/GRU layers]
        |
  Output at EACH timestep: (batch, seq_len, hidden_size)
        |
  [Linear]  → (batch, seq_len, vocab_size=65)
        |
  CrossEntropyLoss per character position
```

**Key difference from forecasting**: We predict at EVERY timestep (many-to-many), not just the last one.

In [ ]:
class CharLanguageModel(nn.Module):
    """Character-level language model using RNN/LSTM/GRU."""
    
    SUPPORTED = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}
    
    def __init__(self, model_type, vocab_size, embed_dim, hidden_size,
                 num_layers=1, dropout=0.0):
        super().__init__()
        self.model_type = model_type
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        rnn_cls = self.SUPPORTED[model_type]
        self.rnn = rnn_cls(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, hidden=None):
        # x: (batch, seq_len) of char indices
        emb = self.embedding(x)           # (batch, seq_len, embed_dim)
        out, hidden = self.rnn(emb, hidden)  # (batch, seq_len, hidden_size)
        logits = self.fc(out)              # (batch, seq_len, vocab_size)
        return logits, hidden
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Parameter comparison
print('Parameter counts (embed=64, hidden=256, layers=2):')
for mt in ['RNN', 'LSTM', 'GRU']:
    m = CharLanguageModel(mt, VOCAB_SIZE, embed_dim=64, hidden_size=256, num_layers=2, dropout=0.1)
    print(f'  {mt:5s}: {m.count_parameters():>10,} parameters')

In [ ]:
def generate_text(model, seed_text, length, temperature=1.0, device=DEVICE):
    """Generate text autoregressively from a seed.
    
    Args:
        model: trained CharLanguageModel
        seed_text: string to start generation from
        length: number of characters to generate
        temperature: controls randomness (0.2=conservative, 1.0=normal, 1.5=creative)
    """
    model.eval()
    chars_generated = list(seed_text)
    
    # Encode seed
    input_idx = [char_to_idx.get(c, 0) for c in seed_text]
    x = torch.LongTensor([input_idx]).to(device)
    
    hidden = None
    
    with torch.no_grad():
        # Process seed to get hidden state
        logits, hidden = model(x, hidden)
        
        # Generate character by character
        last_char_idx = input_idx[-1]
        for _ in range(length):
            x_next = torch.LongTensor([[last_char_idx]]).to(device)
            logits, hidden = model(x_next, hidden)
            
            # Apply temperature
            logits = logits[0, -1, :] / temperature
            probs = torch.softmax(logits, dim=0)
            
            # Sample from distribution
            char_idx = torch.multinomial(probs, 1).item()
            chars_generated.append(idx_to_char[char_idx])
            last_char_idx = char_idx
    
    return ''.join(chars_generated)

---
## 5. Training Infrastructure

In [ ]:
def train_language_model(model, train_loader, val_loader, epochs, lr,
                         device=DEVICE, patience=5, clip_grad=1.0,
                         sample_every=5, seed_text='ROMEO:\n'):
    """Train a character language model with periodic text generation samples."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    
    history = {'train_loss': [], 'val_loss': [], 'perplexity': [], 'samples': {}}
    best_val = float('inf')
    best_state = None
    patience_counter = 0
    
    start = time.time()
    
    for epoch in range(epochs):
        # Train
        model.train()
        losses = []
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits, _ = model(x_batch)
            # Reshape for CrossEntropyLoss: (batch*seq_len, vocab) vs (batch*seq_len,)
            loss = criterion(logits.view(-1, VOCAB_SIZE), y_batch.view(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            losses.append(loss.item())
        
        # Validate
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(device), y_val.to(device)
                logits, _ = model(x_val)
                val_loss = criterion(logits.view(-1, VOCAB_SIZE), y_val.view(-1))
                val_losses.append(val_loss.item())
        
        avg_train = np.mean(losses)
        avg_val = np.mean(val_losses)
        perplexity = np.exp(avg_val)
        
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['perplexity'].append(perplexity)
        scheduler.step(avg_val)
        
        # Early stopping
        if avg_val < best_val:
            best_val = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break
        
        # Print progress and generate sample
        if (epoch + 1) % sample_every == 0 or epoch == 0:
            sample = generate_text(model, seed_text, 150, temperature=0.8)
            history['samples'][epoch+1] = sample
            print(f'  Epoch {epoch+1:3d} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | PPL: {perplexity:.1f}')
            print(f'  Sample: {repr(sample[:100])}...')
    
    elapsed = time.time() - start
    if best_state:
        model.load_state_dict(best_state)
        model = model.to(device)
    return history, elapsed

---
## 6. Hyperparameter Tuning

Quick search with reduced epochs.

In [ ]:
TUNING_CONFIGS = [
    {'embed_dim': 32,  'hidden_size': 128, 'num_layers': 1, 'lr': 0.003, 'dropout': 0.0},
    {'embed_dim': 64,  'hidden_size': 256, 'num_layers': 1, 'lr': 0.002, 'dropout': 0.0},
    {'embed_dim': 64,  'hidden_size': 256, 'num_layers': 2, 'lr': 0.002, 'dropout': 0.1},
    {'embed_dim': 64,  'hidden_size': 512, 'num_layers': 1, 'lr': 0.001, 'dropout': 0.0},
    {'embed_dim': 128, 'hidden_size': 256, 'num_layers': 2, 'lr': 0.001, 'dropout': 0.2},
    {'embed_dim': 64,  'hidden_size': 128, 'num_layers': 2, 'lr': 0.003, 'dropout': 0.1},
]

TUNING_EPOCHS = 5
tuning_results = []

for mt in ['RNN', 'LSTM', 'GRU']:
    print(f'\n{"="*50}')
    print(f'Tuning {mt}')
    print(f'{"="*50}')
    
    for i, cfg in enumerate(TUNING_CONFIGS):
        model = CharLanguageModel(
            model_type=mt, vocab_size=VOCAB_SIZE,
            embed_dim=cfg['embed_dim'], hidden_size=cfg['hidden_size'],
            num_layers=cfg['num_layers'], dropout=cfg['dropout']
        )
        
        hist, t = train_language_model(
            model, train_loader, val_loader,
            epochs=TUNING_EPOCHS, lr=cfg['lr'],
            sample_every=100  # suppress samples during tuning
        )
        
        best_val = min(hist['val_loss'])
        best_ppl = np.exp(best_val)
        tuning_results.append({
            'model_type': mt, 'config': i, **cfg,
            'best_val_loss': best_val, 'perplexity': best_ppl,
            'params': model.count_parameters(), 'time': t
        })
        print(f'  Config {i+1}/{len(TUNING_CONFIGS)}: e={cfg["embed_dim"]}, '
              f'h={cfg["hidden_size"]}, L={cfg["num_layers"]} '
              f'-> val={best_val:.4f}, PPL={best_ppl:.1f} ({t:.1f}s)')

tuning_df = pd.DataFrame(tuning_results)

In [ ]:
# Best configs
best_configs = {}
print('\nBest configurations:')
for mt in ['RNN', 'LSTM', 'GRU']:
    subset = tuning_df[tuning_df['model_type'] == mt]
    best = subset.loc[subset['best_val_loss'].idxmin()]
    best_configs[mt] = best.to_dict()
    print(f'  {mt}: embed={int(best["embed_dim"])}, hidden={int(best["hidden_size"])}, '
          f'layers={int(best["num_layers"])}, PPL={best["perplexity"]:.1f}')

---
## 7. Final Training & Comparison

In [ ]:
FINAL_EPOCHS = 30
final_models = {}
final_histories = {}
final_times = {}

SEED_TEXT = 'ROMEO:\nO, '

for mt in ['RNN', 'LSTM', 'GRU']:
    print(f'\n{"="*60}')
    print(f'Final training: {mt}')
    print(f'{"="*60}')
    cfg = best_configs[mt]
    
    model = CharLanguageModel(
        model_type=mt, vocab_size=VOCAB_SIZE,
        embed_dim=int(cfg['embed_dim']), hidden_size=int(cfg['hidden_size']),
        num_layers=int(cfg['num_layers']), dropout=cfg['dropout']
    )
    
    hist, elapsed = train_language_model(
        model, train_loader, val_loader,
        epochs=FINAL_EPOCHS, lr=cfg['lr'],
        sample_every=5, seed_text=SEED_TEXT
    )
    
    final_models[mt] = model
    final_histories[mt] = hist
    final_times[mt] = elapsed

In [ ]:
# Training curves
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

for mt in ['RNN', 'LSTM', 'GRU']:
    h = final_histories[mt]
    ax1.plot(h['train_loss'], label=mt, color=COLORS[mt], linewidth=1.5)
    ax2.plot(h['val_loss'], label=mt, color=COLORS[mt], linewidth=1.5)
    ax3.plot(h['perplexity'], label=mt, color=COLORS[mt], linewidth=1.5)

ax1.set_title('Training Loss (CE)'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.set_title('Validation Loss (CE)'); ax2.legend(); ax2.set_xlabel('Epoch')
ax3.set_title('Perplexity'); ax3.legend(); ax3.set_xlabel('Epoch')
plt.suptitle('Training Curves — Character Language Models', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature comparison grid
temperatures = [0.2, 0.5, 0.8, 1.0, 1.5]
seed = 'HAMLET:\nTo be, or '

print('TEMPERATURE COMPARISON')
print('Seed:', repr(seed))
print('='*80)

for mt in ['RNN', 'LSTM', 'GRU']:
    print(f'\n--- {mt} ---')
    for temp in temperatures:
        generated = generate_text(final_models[mt], seed, 200, temperature=temp)
        # Show only the generated part (after seed)
        new_text = generated[len(seed):]
        # Clean up for display
        display_text = new_text[:120].replace('\n', ' | ')
        print(f'  T={temp:.1f}: {display_text}')

In [ ]:
# Side-by-side generation comparison (same seed, same temperature)
print('='*80)
print('SIDE-BY-SIDE COMPARISON (Temperature=0.8)')
print('='*80)

seeds = ['ROMEO:\nIs it ', 'JULIET:\nMy love, ', 'Enter MACBETH.\n\n']

for seed in seeds:
    print(f'\nSeed: {repr(seed)}')
    print('-'*70)
    for mt in ['RNN', 'LSTM', 'GRU']:
        generated = generate_text(final_models[mt], seed, 200, temperature=0.8)
        new_text = generated[len(seed):150+len(seed)].replace('\n', ' | ')
        print(f'  {mt:5s}: {new_text}')

In [ ]:
# Final metrics table
final_metrics = {}
for mt in ['RNN', 'LSTM', 'GRU']:
    h = final_histories[mt]
    final_metrics[mt] = {
        'Best Val Loss': min(h['val_loss']),
        'Best Perplexity': min(h['perplexity']),
        'Final Perplexity': h['perplexity'][-1],
        'Parameters': final_models[mt].count_parameters(),
        'Training Time (s)': f'{final_times[mt]:.1f}',
        'Epochs': len(h['val_loss'])
    }

metrics_df = pd.DataFrame(final_metrics).T
for col in ['Best Val Loss', 'Best Perplexity', 'Final Perplexity']:
    metrics_df[col] = metrics_df[col].apply(lambda x: f'{x:.4f}' if isinstance(x, float) else x)

print('\nFINAL COMPARISON')
print('='*80)
metrics_df

---
## 8. Analysis & Insights

In [ ]:
# Training progression: show samples at different epochs for LSTM
print('TRAINING PROGRESSION — LSTM')
print('='*80)
print('How the model learns to write Shakespeare over training epochs:\n')

lstm_samples = final_histories['LSTM'].get('samples', {})
for epoch, sample in sorted(lstm_samples.items()):
    display = sample[len(SEED_TEXT):len(SEED_TEXT)+120].replace('\n', ' | ')
    print(f'Epoch {epoch:3d}: {display}')

print('\nObservation: Early epochs produce gibberish. Later epochs learn spelling,')
print('capitalization, line breaks, and even character names.')

In [ ]:
# Temperature effect visualization
# Generate multiple samples at each temperature and measure character diversity
def measure_diversity(model, seed, length, temperature, n_samples=5):
    unique_chars_list = []
    for _ in range(n_samples):
        text = generate_text(model, seed, length, temperature)
        new_text = text[len(seed):]
        unique_chars_list.append(len(set(new_text)))
    return np.mean(unique_chars_list)

temps = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.2, 1.5, 2.0]
diversity_data = []

for mt in ['RNN', 'LSTM', 'GRU']:
    for t in temps:
        div = measure_diversity(final_models[mt], 'ROMEO:\n', 200, t, n_samples=3)
        diversity_data.append({'model': mt, 'temperature': t, 'unique_chars': div})

div_df = pd.DataFrame(diversity_data)

fig, ax = plt.subplots(figsize=(10, 5))
for mt in ['RNN', 'LSTM', 'GRU']:
    sub = div_df[div_df['model'] == mt]
    ax.plot(sub['temperature'], sub['unique_chars'], 'o-', color=COLORS[mt], label=mt, linewidth=2)

ax.set_xlabel('Temperature', fontsize=12)
ax.set_ylabel('Unique Characters in 200-char Sample', fontsize=12)
ax.set_title('Character Diversity vs Temperature', fontsize=14)
ax.legend()
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5, label='T=1.0')
plt.tight_layout()
plt.show()

print('Low temperature (0.2): Repetitive, conservative, but grammatically better')
print('Medium temperature (0.8): Good balance of creativity and coherence')
print('High temperature (1.5+): Creative but increasingly chaotic')

---
## 9. Key Takeaways

### Character-Level Generation Insights

1. **LSTM/GRU > RNN for generation**: Gated architectures produce more coherent text because they maintain long-range context (remembering who is speaking, maintaining dialogue structure).

2. **Temperature is crucial**: Low temperature → safe but boring; high temperature → creative but chaotic. T=0.7-0.8 is usually the sweet spot.

3. **Character-level models learn structure**: Without any explicit rules, the model learns capitalization, punctuation, line breaks, character names, and play formatting.

4. **Perplexity as a metric**: Lower perplexity = model is less "surprised" by the validation text = better language model.

5. **Training progression is visual**: You can literally watch the model go from random characters → gibberish words → almost-English → recognizable Shakespeare style.

### Extensions
- Word-level model: larger vocabulary but potentially better grammar
- Add attention over the input sequence
- Try with other text corpora (news, code, poetry)
- Beam search decoding instead of random sampling

In [ ]:
# Final summary
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Perplexity
ppl_vals = [min(final_histories[m]['perplexity']) for m in ['RNN', 'LSTM', 'GRU']]
axes[0].bar(['RNN', 'LSTM', 'GRU'], ppl_vals,
            color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']], edgecolor='white')
axes[0].set_title('Best Perplexity (lower = better)', fontsize=13)
for bar, v in zip(axes[0].patches, ppl_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{v:.1f}', ha='center', fontweight='bold')

# Training time
axes[1].bar(['RNN', 'LSTM', 'GRU'], [final_times[m] for m in ['RNN', 'LSTM', 'GRU']],
            color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']], edgecolor='white')
axes[1].set_title('Training Time (s)', fontsize=13)

# Parameters
axes[2].bar(['RNN', 'LSTM', 'GRU'], [final_models[m].count_parameters() for m in ['RNN', 'LSTM', 'GRU']],
            color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']], edgecolor='white')
axes[2].set_title('Parameters', fontsize=13)

plt.suptitle('Final Summary — Shakespeare Text Generation', fontsize=14)
plt.tight_layout()
plt.show()

print('Notebook complete! Proceed to Notebook 07 for Seq2Seq Math Equations.')